# A small classifier, trained in a notebook

A tiny MLP on synthetic data: Preflight's notebook check-in fixture that really trains (an optimizer step on every batch). No downloads; it runs on a CPU or a GPU.

In [ ]:
import torch
from torch.utils.data import DataLoader, TensorDataset

batch_size = 64
lr = 1e-3
epochs = 3
device = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
torch.manual_seed(0)
x = torch.randn(20_000, 256)
y = (x[:, :8].sum(dim=1, keepdim=True) > 0).float()
loader = DataLoader(TensorDataset(x, y), batch_size=batch_size, shuffle=True, num_workers=0)

In [ ]:
model = torch.nn.Sequential(
    torch.nn.Linear(256, 1024), torch.nn.GELU(), torch.nn.Linear(1024, 1024), torch.nn.GELU(), torch.nn.Linear(1024, 1)
).to(device)
opt = torch.optim.AdamW(model.parameters(), lr=lr)
loss_fn = torch.nn.BCEWithLogitsLoss()

In [ ]:
for epoch in range(epochs):
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        loss = loss_fn(model(xb), yb)
        opt.zero_grad()
        loss.backward()
        opt.step()
    print(f"epoch {epoch}: loss {loss.item():.4f}")